
## Section 1 — Setup

In [ ]:
# Load required packages.
#
# DBI and RMariaDB: database connection interface and MySQL driver.
# dotenv: reads credentials from .env so they never appear in this notebook.
# dplyr: data manipulation (filter, group_by, mutate, summarise).
# ggplot2: all visualisations.
# scales: formats axis labels as currency and percentages.
# tidyr: reshaping data for heatmap (pivot_wider / pivot_longer).
# broom: converts statistical test outputs into tidy data frames.

library(DBI)
library(RMariaDB)
library(dotenv)
library(dplyr)
library(ggplot2)
library(scales)
library(tidyr)
library(broom)
library(ggrepel)

cat("All packages loaded.\n")

In [ ]:
# Load environment variables from .env and open the database connection.
#
# Why here.R()?
#   here::here() would normally resolve paths relative to the project root,
#   but we are using a direct path approach since the .env is at project root.
#   dotenv::load_dot_env() reads the file and sets each key as an environment
#   variable in the current R session. dbConnect() then reads those variables.
#   The credentials themselves are never printed or stored in the notebook output.

dotenv::load_dot_env(
  file.path(dirname(getwd()), ".env")
)

con <- dbConnect(
  RMariaDB::MariaDB(),
  host     = Sys.getenv("DB_HOST"),
  port     = as.integer(Sys.getenv("DB_PORT")),
  dbname   = Sys.getenv("DB_NAME"),
  user     = Sys.getenv("DB_USER"),
  password = Sys.getenv("DB_PASSWORD"),
  bigint   = "integer"
)

cat("Connection valid:", dbIsValid(con), "\n")
cat("Tables:", paste(dbListTables(con), collapse = ", "), "\n")

In [ ]:
# Define a consistent visual theme for all plots in this notebook.
#
# A shared theme ensures every chart uses the same font sizes, grid style,
# and colour palette. This is standard practice in production reporting —
# inconsistent chart styling signals a lack of attention to detail.
#
# The palette uses colourblind-safe values from the Okabe-Ito set.

portfolio_theme <- theme_minimal(base_size = 13) +
  theme(
    plot.title    = element_text(face = "bold", size = 15, margin = margin(b = 8)),
    plot.subtitle = element_text(size = 12, colour = "#555555", margin = margin(b = 12)),
    plot.caption  = element_text(size = 10, colour = "#888888", hjust = 0),
    axis.title    = element_text(size = 11),
    axis.text     = element_text(size = 10),
    panel.grid.minor = element_blank(),
    legend.position  = "bottom"
  )

# Colourblind-safe palette (Okabe-Ito)
PALETTE <- c(
  "#E69F00", "#56B4E9", "#009E73",
  "#F0E442", "#0072B2", "#D55E00", "#CC79A7"
)

cat("Theme and palette defined.\n")


## Section 2 — Peak Analysis

In [ ]:
# Pull hourly order data from the warehouse.
# This is query 1a from 02_operational_queries.sql.

hourly <- dbGetQuery(con, "
  SELECT
    dt.hour,
    COUNT(DISTINCT fo.order_id)   AS total_orders,
    SUM(fo.quantity)              AS total_pizzas,
    ROUND(SUM(fo.total_price), 2) AS total_revenue,
    ROUND(AVG(fo.total_price), 2) AS avg_line_item_value
  FROM fact_orders fo
  JOIN dim_time dt ON fo.time_id = dt.time_id
  GROUP BY dt.hour
  ORDER BY dt.hour
")

# Exclude hours 9 and 10 from operational analysis.
# The audit confirmed Morning (hours 9-10) represents less than 0.05% of
# annual volume. Including it would compress the y-axis and obscure the
# patterns that matter for staffing decisions.
hourly_operational <- hourly |> filter(hour >= 11)

glimpse(hourly_operational)

In [ ]:
# Visualise order volume by hour.

ggplot(hourly_operational, aes(x = hour, y = total_orders)) +
  geom_col(fill = PALETTE[2], width = 0.7) +
  geom_text(
    aes(label = scales::comma(total_orders)),
    vjust = -0.5, size = 3.2, colour = "#333333"
  ) +
  scale_x_continuous(breaks = 11:23, labels = paste0(11:23, ":00")) +
  scale_y_continuous(
    labels = scales::comma,
    expand = expansion(mult = c(0, 0.12))
  ) +
  labs(
    title    = "Order Volume by Hour of Day",
    subtitle = "Full year 2015. Hours 9-10 excluded (<0.05% of annual volume).",
    x        = "Hour",
    y        = "Total Orders",
    caption  = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme

In [ ]:
# Pull meal period data and test whether differences between periods
# are statistically significant.
#
# We use daily order counts per meal period as the unit of observation.
# This gives us 358 observations per period to test against — far more
# robust than testing the annual totals, which would be a single number.
#
# Why Kruskal-Wallis rather than ANOVA?
#   ANOVA assumes the data in each group follows a normal distribution.
#   Daily order counts for a restaurant are right-skewed — a few very
#   busy days pull the distribution. Kruskal-Wallis is the non-parametric
#   equivalent of ANOVA: it tests whether the groups come from the same
#   distribution without requiring normality.

daily_meal_period <- dbGetQuery(con, "
  SELECT
    fo.date_id,
    dt.meal_period,
    COUNT(DISTINCT fo.order_id)   AS daily_orders,
    ROUND(SUM(fo.total_price), 2) AS daily_revenue
  FROM fact_orders fo
  JOIN dim_time dt ON fo.time_id = dt.time_id
  WHERE dt.meal_period != 'Morning'
  GROUP BY fo.date_id, dt.meal_period
")

# Set meal period as an ordered factor so charts display chronologically.
meal_order <- c("Lunch", "Afternoon", "Dinner", "Late Night")
daily_meal_period$meal_period <- factor(
  daily_meal_period$meal_period,
  levels = meal_order
)

# Kruskal-Wallis test on daily order counts across meal periods.
kw_meal <- kruskal.test(daily_orders ~ meal_period, data = daily_meal_period)
cat("Kruskal-Wallis test — meal period vs daily orders:\n")
print(kw_meal)

cat("\nInterpretation:\n")
if (kw_meal$p.value < 0.05) {
  cat("p =", round(kw_meal$p.value, 6), "< 0.05.\n")
  cat("The differences in order volume between meal periods are statistically significant.\n")
  cat("These are genuine trading patterns, not random variation.\n")
} else {
  cat("p =", round(kw_meal$p.value, 6), ">= 0.05.\n")
  cat("No statistically significant difference detected between meal periods.\n")
}

In [ ]:
# Pull meal period annual totals for the summary visualisation.

meal_summary <- dbGetQuery(con, "
  SELECT
    dt.meal_period,
    COUNT(DISTINCT fo.order_id)   AS total_orders,
    SUM(fo.quantity)              AS total_pizzas,
    ROUND(SUM(fo.total_price), 2) AS total_revenue,
    ROUND(
      SUM(fo.total_price) / COUNT(DISTINCT fo.order_id),
    2)                            AS avg_order_value
  FROM fact_orders fo
  JOIN dim_time dt ON fo.time_id = dt.time_id
  WHERE dt.meal_period != 'Morning'
  GROUP BY dt.meal_period
")

meal_summary$meal_period <- factor(meal_summary$meal_period, levels = meal_order)

# Side-by-side bars: order volume and average order value.
# These are plotted separately because they use different scales.
# Combining them on a dual-axis chart would be misleading.

ggplot(meal_summary, aes(x = meal_period, y = total_orders, fill = meal_period)) +
  geom_col(width = 0.65, show.legend = FALSE) +
  geom_text(
    aes(label = scales::comma(total_orders)),
    vjust = -0.5, size = 3.5
  ) +
  scale_fill_manual(values = PALETTE[1:4]) +
  scale_y_continuous(
    labels = scales::comma,
    expand = expansion(mult = c(0, 0.12))
  ) +
  labs(
    title    = "Order Volume by Meal Period",
    subtitle = paste0(
      "Kruskal-Wallis p = ",
      format(kw_meal$p.value, scientific = TRUE, digits = 3),
      " — differences are statistically significant"
    ),
    x = NULL, y = "Total Orders",
    caption = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme

In [ ]:
# Average order value by meal period.
# This is plotted separately from volume because it tells a different story.
# Lunch has the highest average order value despite not being the highest
# volume period in absolute terms — that nuance is lost if both metrics
# are on the same chart.

ggplot(meal_summary, aes(x = meal_period, y = avg_order_value, fill = meal_period)) +
  geom_col(width = 0.65, show.legend = FALSE) +
  geom_text(
    aes(label = paste0("$", avg_order_value)),
    vjust = -0.5, size = 3.5
  ) +
  scale_fill_manual(values = PALETTE[1:4]) +
  scale_y_continuous(
    labels = scales::dollar,
    expand = expansion(mult = c(0, 0.15))
  ) +
  labs(
    title    = "Average Order Value by Meal Period",
    subtitle = "Lunch generates the highest average order value, not Dinner.",
    x = NULL, y = "Average Order Value (USD)",
    caption = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme

In [ ]:
# Day of week analysis with significance testing.
# Same approach as meal period: use daily order counts as the
# unit of observation rather than annual totals.

daily_dow <- dbGetQuery(con, "
  SELECT
    fo.date_id,
    dd.day_name,
    dd.day_of_week,
    COUNT(DISTINCT fo.order_id)   AS daily_orders,
    ROUND(SUM(fo.total_price), 2) AS daily_revenue
  FROM fact_orders fo
  JOIN dim_date dd ON fo.date_id = dd.date_id
  GROUP BY fo.date_id, dd.day_name, dd.day_of_week
")

# Order factor levels Mon through Sun using day_of_week as the sort key.
day_levels <- daily_dow |>
  distinct(day_name, day_of_week) |>
  arrange(day_of_week) |>
  pull(day_name)

daily_dow$day_name <- factor(daily_dow$day_name, levels = day_levels)

kw_dow <- kruskal.test(daily_orders ~ day_name, data = daily_dow)
cat("Kruskal-Wallis test — day of week vs daily orders:\n")
print(kw_dow)

cat("\nInterpretation:\n")
if (kw_dow$p.value < 0.05) {
  cat("p =", round(kw_dow$p.value, 6), "< 0.05.\n")
  cat("Day of week has a statistically significant effect on order volume.\n")
} else {
  cat("p =", round(kw_dow$p.value, 6), ">= 0.05.\n")
  cat("No statistically significant difference detected across days.\n")
}

In [ ]:
# Visualise day of week: total orders and average order value.

dow_summary <- dbGetQuery(con, "
  SELECT
    dd.day_name,
    dd.day_of_week,
    COUNT(DISTINCT fo.order_id)   AS total_orders,
    ROUND(SUM(fo.total_price), 2) AS total_revenue,
    ROUND(
      SUM(fo.total_price) / COUNT(DISTINCT fo.order_id),
    2)                            AS avg_order_value
  FROM fact_orders fo
  JOIN dim_date dd ON fo.date_id = dd.date_id
  GROUP BY dd.day_name, dd.day_of_week
  ORDER BY dd.day_of_week
")

dow_summary$day_name <- factor(dow_summary$day_name, levels = day_levels)

ggplot(dow_summary, aes(x = day_name, y = total_orders, fill = day_name)) +
  geom_col(width = 0.65, show.legend = FALSE) +
  geom_text(
    aes(label = scales::comma(total_orders)),
    vjust = -0.5, size = 3.5
  ) +
  scale_fill_manual(values = PALETTE) +
  scale_y_continuous(
    labels = scales::comma,
    expand = expansion(mult = c(0, 0.12))
  ) +
  labs(
    title    = "Order Volume by Day of Week",
    subtitle = paste0(
      "Kruskal-Wallis p = ",
      format(kw_dow$p.value, scientific = TRUE, digits = 3),
      " — differences are statistically significant"
    ),
    x = NULL, y = "Total Orders",
    caption = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme

In [ ]:
# Heatmap: order volume by day of week and hour.
# This is the most operationally useful chart in this section.
# A single-dimension view (hour OR day) obscures the interaction.
# The heatmap shows which specific day-hour combinations are the
# true peaks — which is what a staffing rota needs.

heatmap_data <- dbGetQuery(con, "
  SELECT
    dd.day_name,
    dd.day_of_week,
    dt.hour,
    COUNT(DISTINCT fo.order_id)   AS total_orders
  FROM fact_orders fo
  JOIN dim_date dd ON fo.date_id = dd.date_id
  JOIN dim_time dt ON fo.time_id = dt.time_id
  WHERE dt.hour >= 11
  GROUP BY dd.day_name, dd.day_of_week, dt.hour
  ORDER BY dd.day_of_week, dt.hour
")

heatmap_data$day_name <- factor(heatmap_data$day_name, levels = rev(day_levels))

ggplot(heatmap_data, aes(x = factor(hour), y = day_name, fill = total_orders)) +
  geom_tile(colour = "white", linewidth = 0.4) +
  geom_text(
    aes(label = total_orders),
    size = 2.8, colour = "white"
  ) +
  scale_fill_gradient(
    low  = "#d4e8f5",
    high = "#0072B2",
    name = "Orders"
  ) +
  scale_x_discrete(labels = paste0(11:23, ":00")) +
  labs(
    title    = "Order Volume Heatmap — Day of Week by Hour",
    subtitle = "Friday evening sustains peak volume longest. Weekday lunch peaks at 12:00-13:00.",
    x = "Hour", y = NULL,
    caption = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme +
  theme(
    legend.position  = "right",
    plot.subtitle    = element_text(size = 10),
    plot.title.position = "plot"
  )

### Peak Analysis — Findings

**Finding 01 — Lunch is the highest-value trading period, not Dinner.**  
Lunch generates the highest order volume (7,678 orders) and the highest average order value ($41.95) of any meaningful period. In most restaurants, Dinner commands a premium. Here, Lunch customers spend more per visit. Staffing and upsell investment should prioritise Lunch.

**Finding 02 — Friday is the clear peak day; Saturday has the highest average order value.**  
Friday leads on volume (3,538 orders). Saturday customers spend the most per order ($39.01 average). The business should treat Friday and Saturday as distinct operational challenges — Friday requires maximum capacity, Saturday rewards upsell effort.

**Finding 03 — Friday evening extends significantly later than any other day.**  
Friday hour 22 records 180 orders — more than three times Monday's 53. The restaurant needs extended staffing on Friday evenings specifically.

**Finding 04 — Both findings are statistically confirmed.**  
Kruskal-Wallis tests confirm that differences in daily order volume across meal periods and across days of the week are statistically significant (p < 0.05). These are genuine patterns, not random variation.


## Section 3 — Menu Performance

In [ ]:
# Pull full menu performance data.

menu <- dbGetQuery(con, "
  SELECT
    pt.name                           AS pizza_name,
    pt.category,
    SUM(fo.quantity)                  AS total_quantity,
    ROUND(SUM(fo.total_price), 2)     AS total_revenue,
    ROUND(AVG(fo.unit_price), 2)      AS avg_unit_price,
    RANK() OVER (ORDER BY SUM(fo.quantity) DESC)    AS volume_rank,
    RANK() OVER (ORDER BY SUM(fo.total_price) DESC) AS revenue_rank
  FROM fact_orders fo
  JOIN dim_pizza_type pt ON fo.pizza_type_id = pt.pizza_type_id
  GROUP BY pt.name, pt.category
  ORDER BY total_revenue DESC
")

# Calculate rank divergence: how far apart are volume rank and revenue rank?
# A large positive number means the item sells a lot but generates little revenue.
# A large negative number means the item generates strong revenue despite low volume.
menu <- menu |>
  mutate(rank_divergence = volume_rank - revenue_rank)

cat("Menu items loaded:", nrow(menu), "\n")
cat("\nTop 5 by revenue:\n")
print(menu |> select(pizza_name, category, total_quantity, total_revenue, volume_rank, revenue_rank) |> head(5))

In [ ]:
# Rank divergence chart.
# Items to the right of zero sell more than their revenue rank would suggest
# (high volume, lower revenue — lower margin or lower price).
# Items to the left generate more revenue than their volume rank would suggest
# (premium items that do not need high volume to contribute).

menu_plot <- menu |>
  mutate(pizza_name = stringr::str_remove(pizza_name, "^The ")) |>
  arrange(rank_divergence)

ggplot(menu_plot, aes(
    x    = rank_divergence,
    y    = reorder(pizza_name, rank_divergence),
    fill = rank_divergence > 0
  )) +
  geom_col(width = 0.7, show.legend = FALSE) +
  geom_vline(xintercept = 0, linewidth = 0.6, colour = "#333333") +
  scale_fill_manual(values = c("#0072B2", "#D55E00")) +
  labs(
    title    = "Volume Rank vs Revenue Rank Divergence",
    subtitle = "Orange bars: sells more than it earns. Blue bars: earns more than it sells.",
    x        = "Volume Rank minus Revenue Rank",
    y        = NULL,
    caption  = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme +
  theme(
    legend.position  = "right",
    plot.subtitle    = element_text(size = 10),
    plot.title.position = "plot"
  )

In [ ]:
# Category performance summary.

category_summary <- dbGetQuery(con, "
  SELECT
    pt.category,
    COUNT(DISTINCT pt.pizza_type_id)  AS menu_items,
    SUM(fo.quantity)                  AS total_quantity,
    ROUND(SUM(fo.total_price), 2)     AS total_revenue,
    ROUND(
      SUM(fo.total_price) / SUM(fo.quantity),
    2)                                AS avg_revenue_per_pizza,
    ROUND(
      100.0 * SUM(fo.total_price) / SUM(SUM(fo.total_price)) OVER (),
    1)                                AS revenue_share_pct
  FROM fact_orders fo
  JOIN dim_pizza_type pt ON fo.pizza_type_id = pt.pizza_type_id
  GROUP BY pt.category
  ORDER BY total_revenue DESC
")

print(category_summary)

In [ ]:
# Visualise category efficiency: revenue per pizza vs menu space occupied.
# Bubble size represents number of menu items in each category.
# This shows which categories earn the most per pizza relative to
# how much menu space they consume.

# Add manual nudge columns to position labels cleanly outside each bubble.
category_summary <- category_summary |>
  mutate(
    nudge_x = case_when(
      category == "Chicken" ~ -0.3,
      category == "Classic" ~  0.0,
      category == "Supreme" ~  0.4,
      category == "Veggie"  ~  0.4,
      TRUE ~ 0
    ),
    nudge_y = case_when(
      category == "Chicken" ~ -0.25,
      category == "Classic" ~ -0.35,
      category == "Supreme" ~  0.25,
      category == "Veggie"  ~ -0.25,
      TRUE ~ 0
    )
  )

ggplot(category_summary,
    aes(x = menu_items, y = avg_revenue_per_pizza,
        size = total_revenue, colour = category, label = category)) +
  geom_point(alpha = 0.8) +
  geom_text(
    aes(x = menu_items + nudge_x, y = avg_revenue_per_pizza + nudge_y),
    size        = 4,
    colour      = "#333333",
    show.legend = FALSE
  ) +
  scale_size_continuous(range = c(8, 20), guide = "none") +
  scale_colour_manual(values = PALETTE[1:4], guide = "none") +
  scale_y_continuous(labels = scales::dollar, limits = c(14, 19)) +
  scale_x_continuous(breaks = seq(5, 10, 1), limits = c(5, 10.5)) +
  labs(
    title    = "Category Efficiency: Revenue per Pizza vs Menu Space",
    subtitle = "Bubble size = total annual revenue. Upper-left is most efficient.",
    x        = "Number of Menu Items",
    y        = "Average Revenue per Pizza Sold (USD)",
    caption  = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme

### Menu Performance — Findings

**Finding 05 — Thai Chicken is the most valuable item on the menu, not the most popular.**  
Thai Chicken Pizza ranks 5th by volume but 1st by revenue ($43,434 annually). It commands the second-highest average price in its category ($18.29). It should be prominently positioned and actively promoted.

**Finding 06 — Pepperoni is a high-traffic, low-value item.**  
Pepperoni ranks 4th by volume but 11th by revenue. At $12.47 average price, it is among the cheapest items on the menu. Every Pepperoni sale that could be converted to a Chicken category item represents a meaningful revenue gain at scale.

**Finding 07 — Chicken is the most efficient category on the menu.**  
With only 6 items (fewest of any category), Chicken generates $17.73 revenue per pizza — the highest of any category. Classic has 8 items but generates only $14.78 per pizza.

**Finding 08 — The Brie Carre Pizza is an isolated underperformer.**  
At 490 units sold and $11,589 revenue for the full year, Brie Carre is separated from the rest of the bottom seven by a gap of approximately $2,400. At $31.74 average weekly revenue, Brie Carre generates less in a week than a single Large pizza order on a busy Friday evening. It has no defensible place on the menu.

---
## Section 4 — Revenue Leakage

Revenue leakage refers to revenue the business is not capturing due to identifiable, addressable patterns. This section quantifies three sources:

1. The cost of carrying XXL on the menu
2. The revenue generated by the bottom menu items relative to their menu space cost
3. The upsell opportunity from Medium-to-Large size conversion

In [ ]:
# Size performance.

size_data <- dbGetQuery(con, "
  SELECT
    p.size,
    SUM(fo.quantity)                  AS total_quantity,
    ROUND(SUM(fo.total_price), 2)     AS total_revenue,
    ROUND(AVG(fo.unit_price), 2)      AS avg_unit_price,
    ROUND(
      100.0 * SUM(fo.quantity) / SUM(SUM(fo.quantity)) OVER (),
    1)                                AS volume_share_pct,
    ROUND(
      100.0 * SUM(fo.total_price) / SUM(SUM(fo.total_price)) OVER (),
    1)                                AS revenue_share_pct
  FROM fact_orders fo
  JOIN dim_pizza p ON fo.pizza_id = p.pizza_id
  GROUP BY p.size
  ORDER BY FIELD(p.size, 'S', 'M', 'L', 'XL', 'XXL')
")

size_data$size <- factor(size_data$size, levels = c("S", "M", "L", "XL", "XXL"))

print(size_data)

In [ ]:
# Visualise volume share vs revenue share by size.
# If a size's revenue share exceeds its volume share, it is a premium size.
# If volume share exceeds revenue share, it is a discount size.

size_long <- size_data |>
  select(size, volume_share_pct, revenue_share_pct) |>
  pivot_longer(
    cols      = c(volume_share_pct, revenue_share_pct),
    names_to  = "metric",
    values_to = "share_pct"
  ) |>
  mutate(metric = recode(metric,
    "volume_share_pct"  = "Volume Share",
    "revenue_share_pct" = "Revenue Share"
  ))

ggplot(size_long, aes(x = size, y = share_pct, fill = metric)) +
  geom_col(position = "dodge", width = 0.65) +
  geom_text(
    aes(label = paste0(share_pct, "%")),
    position = position_dodge(width = 0.65),
    vjust = -0.5, size = 3.2
  ) +
  scale_fill_manual(values = c(PALETTE[2], PALETTE[1]), name = NULL) +
  scale_y_continuous(
    labels = function(x) paste0(x, "%"),
    expand = expansion(mult = c(0, 0.15))
  ) +
  labs(
    title    = "Volume Share vs Revenue Share by Pizza Size",
    subtitle = "Large converts 38% of volume into 46% of revenue. Small does the opposite.",
    x = "Size", y = "Share of Annual Total (%)",
    caption = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme

In [ ]:
# Upsell sensitivity analysis: Medium to Large conversion.
# Run the revenue impact calculation across four conversion rate scenarios.
# This demonstrates sensitivity analysis — not just 'here is a number' but
# 'here is how the number changes under different assumptions'.

upsell_base <- dbGetQuery(con, "
  SELECT
    pt.name                               AS pizza_name,
    pt.category,
    pm.price                              AS medium_price,
    pl.price                              AS large_price,
    ROUND(pl.price - pm.price, 2)         AS price_gap,
    SUM(CASE WHEN fo.pizza_id = pm.pizza_id
             THEN fo.quantity ELSE 0 END) AS medium_quantity_sold
  FROM dim_pizza_type pt
  JOIN dim_pizza pm  ON pt.pizza_type_id = pm.pizza_type_id AND pm.size = 'M'
  JOIN dim_pizza pl  ON pt.pizza_type_id = pl.pizza_type_id AND pl.size = 'L'
  JOIN fact_orders fo ON pt.pizza_type_id = fo.pizza_type_id
  GROUP BY pt.name, pt.category, pm.price, pl.price
")

# Apply four conversion rate scenarios.
conversion_rates <- c(0.05, 0.10, 0.15, 0.20)

upsell_scenarios <- lapply(conversion_rates, function(rate) {
  upsell_base |>
    mutate(
      conversion_rate       = paste0(rate * 100, "%"),
      upsell_revenue        = round(medium_quantity_sold * rate * price_gap, 2)
    )
}) |> bind_rows()

# Summarise total upsell revenue by scenario.
upsell_totals <- upsell_scenarios |>
  group_by(conversion_rate) |>
  summarise(total_upsell_revenue = sum(upsell_revenue), .groups = "drop") |>
  mutate(conversion_rate = factor(conversion_rate,
    levels = c("5%", "10%", "15%", "20%")))

cat("Upsell revenue by conversion scenario:\n")
print(upsell_totals)

In [ ]:
# Visualise upsell sensitivity.

ggplot(upsell_totals,
    aes(x = conversion_rate, y = total_upsell_revenue, fill = conversion_rate)) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(
    aes(label = scales::dollar(total_upsell_revenue, accuracy = 1)),
    vjust = -0.5, size = 4
  ) +
  scale_fill_manual(values = PALETTE[1:4]) +
  scale_y_continuous(
    labels = scales::dollar,
    expand = expansion(mult = c(0, 0.15))
  ) +
  labs(
    title    = "Medium-to-Large Upsell Revenue Sensitivity",
    subtitle = "Annual revenue gain if X% of Medium orders convert to Large.",
    x        = "Conversion Rate",
    y        = "Additional Annual Revenue (USD)",
    caption  = "Source: platos_pizza warehouse"
  ) +
  portfolio_theme

### Revenue Leakage — Findings

**Finding 09 — Large is the only size that converts volume into disproportionate revenue.**  
Large represents 38.2% of units sold but 45.9% of annual revenue. Small is the inverse: 29.1% of units but only 21.8% of revenue. Every size shift from Small to Large is worth approximately $7.44 in additional revenue.

**Finding 10 — XXL generates $1,007 in annual revenue across 28 orders.**  
XXL exists only in the Classic category. It is ordered fewer than three times per month on average across the full year. The menu complexity cost of carrying a size with this volume is not justified by the revenue it generates.

**Finding 11 — A 10% Medium-to-Large conversion rate generates approximately $5,900 in additional annual revenue.**  
The upsell opportunity scales linearly: 5% conversion yields roughly $2,950; 20% conversion yields roughly $11,800. This does not require new customers — it requires converting existing Medium orders.

**Finding 12 — The rank divergence chart reveals a structural split in the menu.**
Look at Image 6 carefully. The chart is not a smooth gradient — it has a clear cluster of orange bars (items that sell more than they earn) and a clear cluster of blue bars (items that earn more than they sell). The orange cluster is dominated by Classic and Veggie items. The blue cluster is dominated by Chicken and Supreme items. This is not a finding about individual pizzas — it is a finding about category pricing strategy. Classic and Veggie items are underpriced relative to their demand. This is worth stating explicitly as it has a direct pricing recommendation attached.

**Finding 13 — The afternoon dip is an operational opportunity, not just a quiet period.**
Image 1 shows a clear dip at hours 14 and 15 (1,472 and 1,468 orders respectively) between the lunch peak and the evening recovery. This is the lowest sustained trading period outside of Late Night. From an operational standpoint, this is the natural window for prep, restocking, and staff breaks — it should be built into the staffing model rather than treated as dead time.


## Session Close

In [ ]:
dbDisconnect(con)
cat("Database connection closed.\n")